# Setting up dependencies

In [1]:
%pip uninstall --yes "tensorflow" "matplotlib" "keras" "scikit-learn"

Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0
Found existing installation: matplotlib 3.10.0
Uninstalling matplotlib-3.10.0:
  Successfully uninstalled matplotlib-3.10.0
Found existing installation: keras 3.10.0
Uninstalling keras-3.10.0:
  Successfully uninstalled keras-3.10.0
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
Note: you may need to restart the kernel to use updated packages.


In [2]:
!sudo ln -sf /usr/local/nvidia/lib64/libcuda.so.1 /usr/local/cuda/lib64/libcuda.so

In [3]:
import warnings
warnings.simplefilter('ignore')

In [4]:
import os
import sys
import subprocess

In [5]:
def set_env(input_archive, temp_dir):
    
    wheels_dir = f'{temp_dir}/wheels'
    if not os.path.exists(wheels_dir):
        os.makedirs(temp_dir, exist_ok=True)
        
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'install', 
        '--no-index', 
        '--find-links', 
        f'{temp_dir}/wheels', 
        'unsloth', 
        'trl', 
        'vllm', 
        #'openai_harmony'
    ], check=True)

In [6]:
set_env(
    input_archive='/kaggle/input/notebooks/shelterw/aimo-3-vllm-v16/wheels.tar.gz', 
    temp_dir='/kaggle/tmp/setup'
)

Looking in links: /kaggle/tmp/setup/wheels
Processing /kaggle/tmp/setup/wheels/unsloth-2026.2.1-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/trl-0.24.0-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/vllm-0.16.1rc1.dev136+g7e9149d9a-cp38-abi3-manylinux_2_31_x86_64.whl
Processing /kaggle/tmp/setup/wheels/unsloth_zoo-2026.2.1-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/tyro-1.0.8-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/xformers-0.0.35-py39-none-manylinux_2_28_x86_64.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/datasets-4.3.0-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/transformers-4.57.6-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/protobuf-6.33.5-cp39-abi3-manylinux2014_x86_64.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/prometheus_fastapi_instrumentato

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
keras-tuner 1.4.8 requires keras, which is not installed.
pyldavis 3.4.1 requires scikit-learn>=1.0.0, which is not installed.
ydata-profiling 4.18.1 requires matplotlib<=3.10,>=3.5, which is not installed.
librosa 0.11.0 requires scikit-learn>=1.1.0, which is not installed.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
bigframes 2.31.0 requires matplotlib>=3.7.1, which is not installed.
shap 0.50.0 requires scikit-learn, which is not installed.
pynndescent 0.6.0 requires scikit-learn>=0.18, which is not installed.
umap-learn 0.5.11 requires scikit-learn>=1.6, which is not installed.
sentence-transformers 5.2.0 requires scikit-learn, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
arviz 0.22.0 require

In [7]:
import torch
assert torch.__version__ == "2.10.0+cu128", (f"Torch version is {torch.__version__} instead of 2.8.0+cu128")
assert torch.cuda.is_available() and torch.cuda.device_count() >= 1, "GPU not enabled"

In [8]:
print(f"Torch version: {torch.__version__}")
print(f"Is CUDA compiled in: {torch.version.cuda}")

Torch version: 2.10.0+cu128
Is CUDA compiled in: 12.8


In [9]:
import sys
print(f"Python: {sys.version.split()[0]}")
print(f"Torch Version: {torch.__version__}")

# Check 1: Is Torch even capable of seeing GPUs?
cuda_compiled = torch.version.cuda
print(f"CUDA compiled in Torch: {cuda_compiled}")

# Check 2: Does the OS see the hardware?
available = torch.cuda.is_available()
print(f"Is CUDA available: {available}")

# Check 3: How many GPUs?
count = torch.cuda.device_count()
print(f"GPU Count: {count}")

if count > 0:
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

# Final verification for AIMO 3
if not (available and count >= 1):
    print("CRITICAL: GPU NOT DETECTED. Check Kaggle Accelerator settings.")
else:
    print("SUCCESS: GPU is active and ready for vLLM.")

Python: 3.12.12
Torch Version: 2.10.0+cu128
CUDA compiled in Torch: 12.8
Is CUDA available: True
GPU Count: 2
GPU Name: Tesla T4
SUCCESS: GPU is active and ready for vLLM.


In [10]:
import numpy as np
assert np.__version__ == "2.0.2", (f"Numpy version is {np.__version__} instead of 2.2.0")

# Importing dependencies 

In [11]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"

In [12]:
import time
import warnings
import re
import tempfile
import subprocess
from collections import Counter, defaultdict
from typing import Optional

# Data Processing
import pandas as pd
import polars as pl

# LLM Inference
from transformers import set_seed
from vllm import LLM, SamplingParams
import kaggle_evaluation.aimo_3_inference_server

#fixed seed to get similar score
set_seed(42)
pd.set_option('display.max_colwidth', None)
cutoff_time = time.time() + (4 * 60 + 45) * 60

warnings.simplefilter('ignore')

## Constants

In [13]:
#LLM_MODEL_PATH = "/kaggle/input/models/qwen-lm/qwen-3-5/transformers/qwen3.5-27b/1"
LLM_MODEL_PATH = "/kaggle/input/models/mistral-ai/ministral-3/transformers/ministral-3-3b-instruct-2512/1"
if not os.path.exists(LLM_MODEL_PATH):
    print(f"Error: Path {LLM_MODEL_PATH} not found!")

### Setting environment variables
The following is necessary to make the devices visible for inference

In [14]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# VLLM setup
Here we set up vllm

In [15]:
gpu_name = torch.cuda.get_device_name(0)
if "H100" in gpu_name:
    # High-performance submission settings
    TP_SIZE = 1
    DTYPE = "bfloat16"
    MEM_UTIL = 0.96
elif "T4" in gpu_name:
    # Low-cost testing settings
    TP_SIZE = 2 # Use both T4s to get 32GB VRAM
    DTYPE = "float16" # T4 does not support bfloat16
    MEM_UTIL = 0.85
else:
    raise RuntimeError("Unsupported GPU for vLLM")

llm = LLM(
    LLM_MODEL_PATH,
    dtype=DTYPE,
    max_num_seqs=256,
    max_model_len=4096,       
    trust_remote_code=True,     
    tensor_parallel_size=1,      
    gpu_memory_utilization=MEM_UTIL, 
)

INFO 03-09 16:03:20 [utils.py:229] non-default args: {'trust_remote_code': True, 'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.85, 'max_num_seqs': 256, 'disable_log_stats': True, 'model': '/kaggle/input/models/mistral-ai/ministral-3/transformers/ministral-3-3b-instruct-2512/1'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 03-09 16:03:37 [model.py:530] Resolved architecture: PixtralForConditionalGeneration
WARNING 03-09 16:03:37 [model.py:1891] Casting torch.bfloat16 to torch.float16.
INFO 03-09 16:03:37 [model.py:1553] Using max model len 4096
INFO 03-09 16:03:38 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-09 16:03:38 [vllm.py:747] Asynchronous scheduling is enabled.


[2026-03-09 16:03:38] INFO tekken.py:195: Non special vocabulary size is 130072 with 1000 special tokens.
[2026-03-09 16:03:39] INFO tekken.py:195: Non special vocabulary size is 130072 with 1000 special tokens.


WARNING 03-09 16:03:41 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore_DP0 pid=133) INFO 03-09 16:03:57 [core.py:101] Initializing a V1 LLM engine (v0.16.1rc1.dev136+g7e9149d9a) with config: model='/kaggle/input/models/mistral-ai/ministral-3/transformers/ministral-3-3b-instruct-2512/1', speculative_config=None, tokenizer='/kaggle/input/models/mistral-ai/ministral-3/transformers/ministral-3-3b-instruct-2512/1', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=fp8, enforce_eager=False, enable_return_routed_exper

...

(EngineCore_DP0 pid=133) WARNING 03-09 16:04:01 [weight_utils.py:1261] Found q_scale in the checkpoint (e.g. layers.0.self_attn.attn.q_scale), but not found the expected name in the model (e.g. layers.0.self_attn.attn.attn.q_scale). q_scale is not loaded.
(EngineCore_DP0 pid=133) WARNING 03-09 16:04:02 [weight_utils.py:1261] Found q_scale in the checkpoint (e.g. layers.1.self_attn.attn.q_scale), but not found the expected name in the model (e.g. layers.1.self_attn.attn.attn.q_scale). q_scale is not loaded.
(EngineCore_DP0 pid=133) WARNING 03-09 16:04:03 [weight_utils.py:1261] Found q_scale in the checkpoint (e.g. layers.10.self_attn.attn.q_scale), but not found the expected name in the model (e.g. layers.10.self_attn.attn.attn.q_scale). q_scale is not loaded.
(EngineCore_DP0 pid=133) WARNING 03-09 16:04:03 [weight_utils.py:1261] Found q_scale in the checkpoint (e.g. layers.11.self_attn.attn.q_scale), but not found the expected name in the model (e.g. layers.11.self_attn.attn.attn.q_sca

Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:29<00:00, 29.97s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:29<00:00, 29.97s/it]
(EngineCore_DP0 pid=133) 


(EngineCore_DP0 pid=133) INFO 03-09 16:04:31 [default_loader.py:293] Loading weights took 30.09 seconds
(EngineCore_DP0 pid=133) WARNING 03-09 16:04:31 [marlin_utils_fp8.py:97] Your GPU does not have native support for FP8 computation but FP8 quantization is being used. Weight-only FP8 compression will be used leveraging the Marlin kernel. This may degrade performance for compute-heavy workloads.
(EngineCore_DP0 pid=133) INFO 03-09 16:04:32 [gpu_model_runner.py:4285] Model loading took 4.44 GiB memory and 31.421878 seconds
(EngineCore_DP0 pid=133) INFO 03-09 16:04:32 [gpu_model_runner.py:5207] Encoder cache will be initialized with a budget of 8192 tokens, and profiled with 2 image items of the maximum feature size.
(EngineCore_DP0 pid=133) WARNING 03-09 16:04:33 [context.py:305] PixtralProcessorAdapter did not return `BatchFeature`. Make sure to match the behaviour of `ProcessorMixin` when implementing custom processors.
(EngineCore_DP0 pid=133) INFO 03-09 16:04:45 [backends.py:916] U

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:04<00:00, 11.01it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 12.86it/s]


(EngineCore_DP0 pid=133) INFO 03-09 16:06:34 [gpu_model_runner.py:5313] Graph capturing finished in 8 secs, took 0.75 GiB
(EngineCore_DP0 pid=133) INFO 03-09 16:06:34 [core.py:282] init engine (profile, create kv cache, warmup model) took 122.51 seconds
INFO 03-09 16:06:35 [llm.py:382] Supported tasks: ['generate']


In [16]:
tokenizer = llm.get_tokenizer()

In [17]:
sampling_params = SamplingParams(
    temperature=1.0,
    min_p=0.01,
    skip_special_tokens=True,     
    max_tokens=32768,
)

# Trial inference

In [18]:
# 1. Run a simple test prompt
test_prompt = "The capital of France is"

# 2. Generate
outputs = llm.generate([test_prompt], sampling_params)

# 3. Print results
for output in outputs:
    print(f"Prompt: {output.prompt}")
    print(f"Output: {output.outputs[0].text}")

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Prompt: The capital of France is
Output:  Paris, a city known for its rich history, art, fashion, and famous sites such as the Eiffel Tower, Louvre Museum, Notre-Dame Cathedral, and Arc de Triomphe. Here is some more information about the capital city for various educational purposes:

1. **Geography:**
   - Located along the River Seine.
   - The city area stretches over 105 square kilometers.
   - Home to famous parks: Luxembourg Gardens, Tuileries Garden, and Bois de Boulogne.

2. **History:**
   - Founded by the Romans in 250 BC.
   - Became the capital during the French Revolution in 1789.
   - Was one of the earliest centers of the Western Christian Church, playing a pivotal role in the Catholic Reformation.

3. **Culture and Arts:**
   - A mecca for the arts: 45 UNESCO World Heritage Sites and 39 museums.
   - Major art forms include ballet, opera, and painting, with contributions from artists like Eugène Delacroix, Pierre-Auguste Renoir, and Monet.
   - Famous landmarks include